# Sistema Automatizado de Coleta de Comentários no Google Maps

<!-- ### Informações importantes -->
Este projeto apresenta o desenvolvimento de um sistema computacional capaz de realizar a **coleta automatizada de comentários e avaliações disponíveis no Google Maps**.  
A solução foi implementada utilizando **Selenium WebDriver** para navegação automatizada, extraindo informações como identificador do comentário, usuário, data de publicação, texto, avaliação em estrelas, presença de foto e status de *Local Guide*.  

Os dados coletados são organizados em uma base estruturada e exportados em formato compatível com planilhas, permitindo sua posterior utilização em análises quantitativas e qualitativas.  
O sistema busca **otimizar o processo de obtenção de informações** em grande escala, reduzindo a necessidade de coleta manual, garantindo maior eficiência, confiabilidade e padronização dos resultados.

## 📖 Utilização
Para utilizar o sistema, é necessário fornecer o **link do local desejado no Google Maps**.  
O programa acessará automaticamente a página indicada, realizará a extração dos comentários disponíveis e os salvará em uma planilha estruturada.  

Etapas básicas de uso:
1. Informar o link do local do Google Maps a ser analisado.  
1. Executar o notebook.  
1. Aguardar a coleta automatizada dos comentários.  
1. Consultar os dados exportados na planilha gerada.

### Insira no campo abaixo, entre as aspas duplas a URL do local do Google Maps

In [3]:
web_scrapping_target = "https://maps.app.goo.gl/1DE8iiLbRtKb2vJB8?g_st=aw"
# web_scrapping_target = "https://www.google.com/maps/place/Posto+Ipiranga/@-23.6185106,-46.6071845,17.25z/data=!4m16!1m9!3m8!1s0x94ce5bf54ea3c9c9:0xf530f12d6d967cf9!2sAssa%C3%AD+Atacadista!8m2!3d-23.6183251!4d-46.6092761!9m1!1b1!16s%2Fg%2F11mxzpwghk!3m5!1s0x94ce5bc0925106a7:0x4a46fd8e45009475!8m2!3d-23.6180355!4d-46.6052243!16s%2Fg%2F1yfj9_h05?entry=ttu&g_ep=EgoyMDI1MDkwMi4wIKXMDSoASAFQAw%3D%3D"


## Coleta automática

### Importações

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

import urllib.parse
import re
import pandas as pd
import os

from datetime import datetime, timedelta
from time import sleep

### Definição de variáveis globais e Funções

In [10]:
# Instanciar navegador
navegador = webdriver.Edge()

# definir tempo máximo de aguardo para espera esplícita (explicit wait)
wait = WebDriverWait(navegador, 10)

colunas_df = ["id", "usuario", "ano_postagem", "comentario", "avaliacao", "foto", "local_guide"]

In [47]:
# funções
def coletar_dados_comentário(elemento_pai: WebElement):
    id = elemento_pai.get_attribute('data-review-id')
    usuario = elemento_pai.find_element(By.CSS_SELECTOR, '.d4r55').text
    avaliacao = elemento_pai.find_element(By.CSS_SELECTOR, '.kvMYJc').get_attribute('aria-label')
    texto_tempo_postagem = elemento_pai.find_element(By.CSS_SELECTOR, '.rsqaWe').text

    ano_postagem = aproximar_ano(texto_tempo_postagem, datetime.now())

    # Expandir botão de comentário (se tiver)
    while True:
        botao_expandir = elemento_pai.find_elements(By.CSS_SELECTOR, '.w8nwRe.kyuRq')
        print(botao_expandir)
        if botao_expandir == []:
            break
        else:
            wait.until(EC.element_to_be_clickable((botao_expandir[0]))).click()
        # navegador.execute_script("arguments[0].click();", elemento_pai.find_element(By.CSS_SELECTOR, '.w8nwRe.kyuRq'))

        sleep(0.3)
            
    # Pegar comentário
    try:        
        comentario = elemento_pai.find_element(By.CSS_SELECTOR, '.MyEned').text
    except NoSuchElementException:
        comentario = 'Sem comentários.'    

    # Se tem foto
    try: 
        elemento_pai.find_element(By.CSS_SELECTOR, '.KtCyie')
        foto = True
    except NoSuchElementException:
        foto = False
        

    # Local guide
    try:
        if "Local Guide" in elemento_pai.find_element(By.CSS_SELECTOR, '.RfnDt').text:
            local_guide = True
        else:
            local_guide = False
    except NoSuchElementException:
        local_guide = False
        
    return {
        "id": id,
        "usuario": usuario,
        "ano_postagem": ano_postagem,
        "avaliacao": avaliacao,
        "comentario": comentario,
        "local_guide": local_guide,
        "foto": foto
    }


def aproximar_ano(texto: str, referencia: datetime = None) -> int:
    if referencia is None:
        referencia = datetime.now()

    texto = texto.lower().strip()
    texto = texto.replace("editado ", "")  # remove prefixo se existir

    unidades = {
        "ano": 365,
        "anos": 365,
        "mês": 30,
        "meses": 30,
        "semana": 7,
        "semanas": 7,
        "dia": 1,
        "dias": 1
    }

    match = re.match(r"(um|\d+)\s+(\w+)\s+atrás", texto)
    if match:
        quantidade = match.group(1)
        unidade = match.group(2)

        quantidade = 1 if quantidade == "um" else int(quantidade)

        if unidade in unidades:
            dias = quantidade * unidades[unidade]
            data_aproximada = referencia - timedelta(days=dias)
            return data_aproximada.year

    return referencia.year  # fallback: assume que é este ano



def gerar_nome_arquivo(url):
    # Decodifica a URL
    url_decodificada = urllib.parse.unquote(url)

    # Extrai o nome do local
    nome_local_match = re.search(r'/place/([^/@]+)', url_decodificada)
    nome_local = nome_local_match.group(1).replace('+', '').replace(' ', '') if nome_local_match else 'LocalDesconhecido'

    # Extrai o identificador único (place ID)
    id_match = re.search(r'!1s(0x[a-fA-F0-9]+:0x[a-fA-F0-9]+)', url_decodificada)
    place_id = id_match.group(1).replace(':', '_') if id_match else 'IDDesconhecido'

    # Data atual no formato dd-mm-YYYY
    data_atual = datetime.now().strftime('%d-%m-%Y')

    # Monta o nome do arquivo
    nome_arquivo = f"Comentarios_{nome_local}_{place_id}_{data_atual}.xlsx"
    return nome_arquivo

### Execução principal

In [18]:
"""
Aqui inicialmente, o programa acessará a pagina do local escolhido, 
entrará na página de comentários e colocará em ordem dos mais recentes.
"""
# Acessar página
navegador.get(web_scrapping_target)

# entrar no campo de comentários
wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="QA0Szd"]/div/div/div[1]/div[2]/div/div[1]/div/div/div[3]/div/div/button[2]'))).click()

# Aguardar até carregar os comentários
wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="QA0Szd"]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[9][contains(@class, "m6QErb") and contains(@class, "XiKgde")]')))

# Ordernar em mais recentes
wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="QA0Szd"]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[7]/div[2]/button'))).click()
wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="action-menu"]/div/div[2]'))).click()

# Esperar carregar comentários
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.jftiEf')))

proximo_comentario = 0


nome_arquivo = gerar_nome_arquivo(navegador.current_url)
if os.path.exists(nome_arquivo):
    df = pd.read_excel(nome_arquivo)
else:
    df = pd.DataFrame(columns=colunas_df)

In [45]:


stopScrapping = False
for c in range(30):
    stopScrapping = False
    df_temp = pd.DataFrame(columns=colunas_df)
    
    print('Scrapping...')
    elementos_comentarios = navegador.find_elements(By.CSS_SELECTOR, '.jftiEf')
    quantidade_comentarios = len(elementos_comentarios)
    # print(f'Quantidade de comentários: {quantidade_comentarios}')
    
    for i in range(proximo_comentario, quantidade_comentarios): # Rodar os ultimos 10
        print(f'Pegando o comentário {i}')
        # navegador.execute_script("arguments[0].scrollIntoView({block: 'center'});", elementos_comentarios[i])
        dados = coletar_dados_comentário(elementos_comentarios[i])
        
        if dados['ano_postagem'] <= 2016:
            stopScrapping = True
            break

        if dados['id'] not in df['id'].values:
            print('Valor NÃO repetido. adicionar')
            df_temp = pd.concat([df_temp, pd.DataFrame([dados])], ignore_index=True)
        else:
            print('Valor repetido.')
        proximo_comentario = i + 1
        sleep(0.1)

    # Rolar até o final página
    print('Rolando para o final da página')
    navegador.execute_script("arguments[0].scrollIntoView({ block: 'center'});", navegador.find_elements(By.CSS_SELECTOR, '.jftiEf')[-1])
    sleep(0.5)

    # while True:
    #     try:
    #         print('Tentando pegar o elemento load')
    #         elemento_load = navegador.find_element(By.CSS_SELECTOR, '.qjESne')
    #         resp = navegador.execute_script("""
    #             var elem = arguments[0];
    #             var rect = elem.getBoundingClientRect();
    #             var windowHeight = (window.innerHeight || document.documentElement.clientHeight);
    #             var windowWidth = (window.innerWidth || document.documentElement.clientWidth);
        
    #             return (
    #                 rect.bottom > 0 &&
    #                 rect.right > 0 &&
    #                 rect.top < windowHeight &&
    #                 rect.left < windowWidth
    #             );
    #         """, elemento_load)
    try:
        print('Tentando pegar o elemento load')
        elemento_load = navegador.find_element(By.CSS_SELECTOR, '.qjESne')
 
        while True:
            resp = navegador.execute_script("""
                var elem = arguments[0];
                var rect = elem.getBoundingClientRect();
                var windowHeight = (window.innerHeight || document.documentElement.clientHeight);
                var windowWidth = (window.innerWidth || document.documentElement.clientWidth);
        
                return (
                    rect.bottom > 0 &&
                    rect.right > 0 &&
                    rect.top < windowHeight &&
                    rect.left < windowWidth
                );
            """, elemento_load)
    
            if resp == False:
                print('Parou de carregar')
                # Estava carrgando e parou de carregar
                break
            # else:
                print('Carregando...')
            sleep(0.2)
    except NoSuchElementException:
        # NÃO TEM MAIS COMENTÁRIOS - NÃO TEM ELEMENTO DE LOAD
        print('Loading nn encontrado - SEM MAIS COMENTÁRIOS')
        stopScrapping = True
    except StaleElementReferenceException as erro:
        print(erro)
        stopScrapping = True
    else:
        # Se não deu erro, apareceu o elemento de carregar, e ja finalizou
        # Verificar se tem mais dados
        print('Loading concluído')
        
        quantidade_comentarios = len(navegador.find_elements(By.CSS_SELECTOR, '.jftiEf'))
        print(f'Quantidade de comentários: {quantidade_comentarios}')
        print(f'Próximo comentário: indice {proximo_comentario}')
        if quantidade_comentarios > proximo_comentario:
            # Existem mais comentários
            print('Tem mais comentários, continuar')
            pass
        else:
            # NÃO TEM MAIS COMENTÁRIOS
            print('Não tem mais comentários')
            stopScrapping = True


    # Salvar dados temporários nos dados finais
    print('Salvando dados da nova pesquisa na tabela')
    df = pd.concat([df, df_temp], ignore_index=True)


    if stopScrapping:
        print('Stop scrapping')
        break
print('Finalizou loop')

Scrapping...
Pegando o comentário 1733
Valor NÃO repetido. adicionar
Pegando o comentário 1734
Valor NÃO repetido. adicionar
Pegando o comentário 1735
Valor NÃO repetido. adicionar
Pegando o comentário 1736
Valor NÃO repetido. adicionar
Pegando o comentário 1737
Valor NÃO repetido. adicionar
Pegando o comentário 1738
Valor NÃO repetido. adicionar
Pegando o comentário 1739
Valor NÃO repetido. adicionar
Rolando para o final da página
Tentando pegar o elemento load
Parou de carregar
Loading concluído
Quantidade de comentários: 1750
Próximo comentário: indice 1740
Tem mais comentários, continuar
Salvando dados da nova pesquisa na tabela
Scrapping...
Pegando o comentário 1740
Valor NÃO repetido. adicionar
Pegando o comentário 1741
Valor NÃO repetido. adicionar
Pegando o comentário 1742
Valor NÃO repetido. adicionar
Pegando o comentário 1743
Valor NÃO repetido. adicionar
Pegando o comentário 1744
Valor NÃO repetido. adicionar
Pegando o comentário 1745
Valor NÃO repetido. adicionar
Pegando o 

#### Execução Principal

Aqui o código principal é executado, chamando as funções definidas anteriormente.


In [35]:
proximo_comentario = 1730

In [33]:
df

,id,usuario,ano_postagem,comentario,avaliacao,foto,local_guide
0,Ci9DQUlRQUNvZENodHljRjlvT25Cc1VrMWFlbEJuVnpWR2...,Simone Bastos,2025,"Gente , nada contra a estação Sé porque eu sei...",5 estrelas,True,True
1,Ci9DQUlRQUNvZENodHljRjlvT21SQmJqZ3RiMjlLTVdnMF...,Anderson Nunes,2025,"🚉 Sé\n\nCuriosidade: Estação símbolo do metrô,...",3 estrelas,False,True
2,Ci9DQUlRQUNvZENodHljRjlvT2xGSFNqZGtibmRHY25Oem...,WESLEY COSTA,2025,Sem comentários.,4 estrelas,False,True
3,Ci9DQUlRQUNvZENodHljRjlvT25WRFQwNUViV1UxYjFoRU...,Leonardo Moura,2025,Sem comentários.,5 estrelas,False,True
4,Ci9DQUlRQUNvZENodHljRjlvT21wR1VDMVRjR2N5VlVaUk...,Denise Aparecida,2025,Fácil acesso para todos os lugares de São Paulo,5 estrelas,False,True
...,...,...,...,...,...,...,...
1715,ChZDSUhNMG9nS0VJQ0FnSURBMS1pYlZ3EAE,Kryssia Raisse,2018,Sem comentários.,4 estrelas,False,True
1716,ChdDSUhNMG9nS0VJQ0FnSUNndk5pSjJBRRAB,Cleyton Lopes (Cleytinho),2018,Estação que interliga praticamente todas as zo...,5 estrelas,False,True
1717,ChZDSUhNMG9nS0VJQ0FnSURROTVtNVVBEAE,Elka Almeida,2018,Sem comentários.,5 estrelas,False,True
1718,ChZDSUhNMG9nS0VJQ0FnSURRNGJMbVZBEAE,Iara Conca,2018,"A Praça da Sé é maravilhosa, assim com a Cated...",3 estrelas,False,True


### EXPORTAR BASE DE DADOS

In [46]:
display(df)
df.to_excel(nome_arquivo, index=False)

,id,usuario,ano_postagem,comentario,avaliacao,foto,local_guide
0,Ci9DQUlRQUNvZENodHljRjlvT25Cc1VrMWFlbEJuVnpWR2...,Simone Bastos,2025,"Gente , nada contra a estação Sé porque eu sei...",5 estrelas,True,True
1,Ci9DQUlRQUNvZENodHljRjlvT21SQmJqZ3RiMjlLTVdnMF...,Anderson Nunes,2025,"🚉 Sé\n\nCuriosidade: Estação símbolo do metrô,...",3 estrelas,False,True
2,Ci9DQUlRQUNvZENodHljRjlvT2xGSFNqZGtibmRHY25Oem...,WESLEY COSTA,2025,Sem comentários.,4 estrelas,False,True
3,Ci9DQUlRQUNvZENodHljRjlvT25WRFQwNUViV1UxYjFoRU...,Leonardo Moura,2025,Sem comentários.,5 estrelas,False,True
4,Ci9DQUlRQUNvZENodHljRjlvT21wR1VDMVRjR2N5VlVaUk...,Denise Aparecida,2025,Fácil acesso para todos os lugares de São Paulo,5 estrelas,False,True
...,...,...,...,...,...,...,...
1791,ChdDSUhNMG9nS0VJQ0FnSUNBc01XY19BRRAB,Adriana Cunha,2018,Sem comentários.,4 estrelas,False,False
1792,ChZDSUhNMG9nS0VJQ0FnSUNBOVpubkRnEAE,Anderson Pereira,2018,Sem comentários.,5 estrelas,False,True
1793,ChdDSUhNMG9nS0VJQ0FnSUNBcnNqRzFBRRAB,Douglas Prado,2018,Sem comentários.,3 estrelas,False,True
1794,ChdDSUhNMG9nS0VJQ0FnSURBcC1iMDRRRRAB,Marcella Fontes,2018,Cheio de gente\nPonto turístico,3 estrelas,False,True


### 📊 Resultados e Conclusões

Os resultados obtidos nesta execução devem ser interpretados e descritos na monografia.  
Adicione nesta seção suas observações e análises.


In [43]:
gerar_nome_arquivo(navegador.current_url)

'Comentarios_Sé_0x94ce59aa5b004689_0x37c720ec525c8bd9_05-09-2025.xlsx'

In [25]:
coletar_dados_comentário(navegador.find_element(By.XPATH, '//*[@id="QA0Szd"]/div/div/div[1]/div[2]/div/div[1]/div/div/div[2]/div[9]/div[4217]/div'))

{'id': 'ChZDSUhNMG9nS0VJQ0FnSUNpeWFqRGV3EAE',
 'usuario': 'WILLSON SUBSEA',
 'ano_postagem': 2021,
 'avaliacao': '5 estrelas',
 'comentario': 'Excelente bairro, antigo .. muito bom, conducao para todo lugar, vasta opcao , proximo ao centro da cidade e zona sul , zona norte.. tem metro , V.l.t , onibus , trem .. proximo do maracana , rodoviaria novo Rio.. proximo do centro comercial , tem o camelodromo , o Saara ..\ntudo com precos acessiveis ..\nvale muito apena',
 'local_guide': True,
 'foto': False}